# LeetCode 454 — 4Sum II bằng Python

Link đề: https://leetcode.com/problems/4sum-ii/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Counting, Two Pointers

Skeleton LeetCode:

```python
class Solution:
    def fourSumCount(self, nums1: List[int], nums2: List[int], nums3: List[int], nums4: List[int]) -> int:
        ...
```

Ghi chú: đề được viết lại theo hiểu biết về đề gốc; nên đối chiếu lại đề gốc nếu muốn khớp từng chữ. Mọi đáp án trong notebook được kiểm chứng bằng code, so với oracle vét cạn `O(n^4)` độc lập trên input nhỏ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho **bốn** mảng cùng độ dài `n`; đếm số **bộ chỉ số** `(i, j, k, l)` sao cho `nums1[i] + nums2[j] + nums3[k] + nums4[l] == 0`. Đếm theo **chỉ số**, không theo giá trị.
2. Nắm các công cụ Python: vòng `for` lồng nhau, `itertools.product`, `collections.Counter`/`dict` để đếm, biểu thức sinh (generator) trong `sum(...)`, `sorted`, hai con trỏ, `defaultdict`.
3. Đi từ **bốn vòng lặp `O(n^4)`** → **ba vòng + bảng đếm `O(n^3)`** → **gặp nhau ở giữa (meet in the middle) `O(n^2)`** → **nối hai bảng đếm theo khoá** → **sắp xếp + hai con trỏ**.
4. Hiểu mấu chốt: tách `a + b + c + d = 0` thành `(a + b) = -(c + d)`; mỗi vế chỉ phụ thuộc **hai** mảng, nên ta đếm **mọi tổng cặp** của một nửa rồi tra bảng cho nửa còn lại: `O(n^4)` → `O(n^2)`.
5. Tránh các lỗi kinh điển: dùng `set` thay vì đếm (mất số lần), loại trùng giá trị (đề đếm chỉ số), quên dấu trừ khi tra `-(c + d)`, dùng `defaultdict` rồi tra làm phình bảng, và nhân/ cộng sai khi hai vế có nhiều tổng bằng nhau.

## 2. Đề bài & ví dụ

Cho bốn mảng số nguyên `nums1`, `nums2`, `nums3`, `nums4`, **cùng độ dài `n`**. Hãy trả về **số bộ `(i, j, k, l)`** sao cho:

- `0 <= i, j, k, l < n`
- `nums1[i] + nums2[j] + nums3[k] + nums4[l] == 0`

**Ví dụ 1**

- Input: `nums1 = [1,2]`, `nums2 = [-2,-1]`, `nums3 = [-1,2]`, `nums4 = [0,2]`
- Output: `2`
- Giải thích: hai bộ là `(0,0,0,1)`: `1 + (-2) + (-1) + 2 = 0` và `(1,1,0,0)`: `2 + (-1) + (-1) + 0 = 0`.

**Ví dụ 2**

- Input: `nums1 = [0]`, `nums2 = [0]`, `nums3 = [0]`, `nums4 = [0]`
- Output: `1`

**Ràng buộc:**

- `n == nums1.length == nums2.length == nums3.length == nums4.length`
- `1 <= n <= 200`
- `-2^28 <= nums1[i], nums2[i], nums3[i], nums4[i] <= 2^28`

Chú ý: với `n = 200`, `n^4 = 1.6 × 10^9` — quá lớn cho bốn vòng lặp Python; còn `n^2 = 40000` thì rất nhỏ. Đáp án có thể lên tới `n^4` (khi mọi phần tử bằng `0`).

In [1]:
from typing import List
from collections import Counter, defaultdict
from itertools import product
import random
import time


examples = [
    (([1, 2], [-2, -1], [-1, 2], [0, 2]), 2),
    (([0], [0], [0], [0]), 1),
]
for arrays, expected in examples:
    print(arrays, "->", expected)

([1, 2], [-2, -1], [-1, 2], [0, 2]) -> 2
([0], [0], [0], [0]) -> 1


## 3. Kiến thức nền

### 3.1. Đếm **bộ chỉ số**, không phải bộ giá trị

Hai chỉ số khác nhau có thể cùng giá trị. Ví dụ `nums1 = [0, 0]` có **hai** cách chọn phần tử `0`; mỗi cách là một bộ chỉ số khác nhau. Vì thế **không** loại trùng giá trị (khác bài 4Sum gốc, bài 18).

In [2]:
nums1 = [0, 0]
nums2 = nums3 = nums4 = [0]
# mọi (i, 0, 0, 0) đều có tổng 0: có 2 bộ chỉ số dù chỉ có một "giá trị 0"
print(sum(1 for i in range(2) if nums1[i] + 0 + 0 + 0 == 0))    # 2

2


### 3.2. `Counter`/`dict` để đếm số lần xuất hiện

`Counter(iterable)` đếm số lần mỗi giá trị xuất hiện; tra khoá **chưa có** cho `0` mà **không thêm** khoá đó vào bảng. Với `dict` thường, `d[k]` báo `KeyError` nếu thiếu; dùng `d.get(k, 0)`.

In [3]:
cnt = Counter([-1, 0, 0, 1])
print(cnt)                  # Counter({0: 2, -1: 1, 1: 1})
print(cnt[0], cnt[5])       # 2 0
print(len(cnt))             # 3: tra cnt[5] KHÔNG thêm khoá 5

d = {"a": 1}
print(d.get("b", 0))        # 0

Counter({0: 2, -1: 1, 1: 1})
2 0
3
0


### 3.3. Biểu thức sinh (generator expression) trong `sum`

`sum(f(x) for x in iterable)` cộng các giá trị `f(x)` mà không tạo list trung gian. Với hai vòng lồng nhau: `sum(f(c, d) for c in A for d in B)` — vòng `for` đầu là vòng ngoài.

In [4]:
A, B = [1, 2], [10, 20]
print(sum(a * b for a in A for b in B))      # 1*10 + 1*20 + 2*10 + 2*20 = 90
print(Counter(a + b for a in A for b in B))  # Counter({11: 1, 21: 1, 12: 1, 22: 1})

90
Counter({11: 1, 21: 1, 12: 1, 22: 1})


### 3.4. `itertools.product` — tích Descartes

`product(A, B, C, D)` sinh mọi bộ `(a, b, c, d)` — tương đương bốn vòng `for` lồng nhau. Ta dùng làm **oracle** vét cạn gọn gàng.

In [5]:
print(list(product([1, 2], [3, 4])))                       # [(1, 3), (1, 4), (2, 3), (2, 4)]
print(len(list(product(range(3), range(3), range(3), range(3)))))   # 81 = 3^4

[(1, 3), (1, 4), (2, 3), (2, 4)]
81


### 3.5. Hai con trỏ trên hai dãy đã sắp xếp

Cho hai dãy tăng dần `A` và `C`, muốn tìm cặp `A[i] + C[j] == 0`: đặt `i` ở **đầu** `A`, `j` ở **cuối** `C`. Tổng `< 0` → cần lớn hơn → tăng `i`; tổng `> 0` → cần nhỏ hơn → giảm `j`. Mỗi bước loại bỏ một phần tử → tuyến tính. Ta sẽ dùng ở Cách 5 (có xử lý giá trị lặp).

In [6]:
A = [-3, -1, 0, 2, 5]
C = [-5, -2, 1, 3, 4]
i, j, pairs = 0, len(C) - 1, []
while i < len(A) and j >= 0:
    s = A[i] + C[j]
    if s < 0:
        i += 1
    elif s > 0:
        j -= 1
    else:
        pairs.append((A[i], C[j]))
        i += 1
        j -= 1
print(pairs)        # [(-3, 3), (-1, 1), (2, -2), (5, -5)]
assert pairs == [(-3, 3), (-1, 1), (2, -2), (5, -5)]

[(-3, 3), (-1, 1), (2, -2), (5, -5)]


### 3.6. `defaultdict` và cái bẫy "tra là thêm"

`defaultdict(int)` tự tạo khoá với giá trị `0` khi **tra** khoá chưa có. Tiện để đếm (`d[k] += 1`) nhưng nếu **tra cứu** nhiều khoá không tồn tại, bảng phình ra vô ích.

In [7]:
dd = defaultdict(int)
dd["a"] += 1
_ = dd["x"], dd["y"], dd["z"]      # chỉ tra mà thôi...
print(len(dd), dict(dd))           # 4 khoá: bị thêm vào!

cn = Counter({"a": 1})
_ = cn["x"], cn["y"], cn["z"]
print(len(cn))                     # 1: Counter không thêm khi tra

4 {'a': 1, 'x': 0, 'y': 0, 'z': 0}
1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Xét Ví dụ 1: `nums1 = [1,2]`, `nums2 = [-2,-1]`, `nums3 = [-1,2]`, `nums4 = [0,2]`.

Có `2^4 = 16` bộ chỉ số. Ta có thể liệt kê hết nhưng chậm. Thay vào đó **nhóm đôi**:

- Các tổng `a + b` (từ `nums1`, `nums2`): `1+(-2) = -1`, `1+(-1) = 0`, `2+(-2) = 0`, `2+(-1) = 1`. Tức là `{-1: 1 lần, 0: 2 lần, 1: 1 lần}`.
- Các tổng `c + d` (từ `nums3`, `nums4`): `-1+0 = -1`, `-1+2 = 1`, `2+0 = 2`, `2+2 = 4`.

Điều kiện `a + b + c + d = 0` ⇔ `a + b = -(c + d)`. Với mỗi tổng `c + d`, số cách chọn `(a, b)` đúng bằng số lần `-(c + d)` xuất hiện trong bảng tổng `a + b`:

| `c + d` | cần `a + b` | số lần |
|---|---|---|
| `-1` | `1` | 1 |
| `1` | `-1` | 1 |
| `2` | `-2` | 0 |
| `4` | `-4` | 0 |

Tổng: `1 + 1 + 0 + 0 = 2` ✓. Ta kiểm chứng bằng code:

In [8]:
nums1, nums2, nums3, nums4 = [1, 2], [-2, -1], [-1, 2], [0, 2]
ab = Counter(a + b for a in nums1 for b in nums2)
print("tổng a+b:", dict(ab))
total = 0
for c in nums3:
    for d in nums4:
        need = -(c + d)
        print(f"c+d = {c + d:>2} -> cần a+b = {need:>2}: có {ab[need]} cách")
        total += ab[need]
print("tổng cộng:", total)
assert total == 2

tổng a+b: {-1: 1, 0: 2, 1: 1}
c+d = -1 -> cần a+b =  1: có 1 cách
c+d =  1 -> cần a+b = -1: có 1 cách
c+d =  2 -> cần a+b = -2: có 0 cách
c+d =  4 -> cần a+b = -4: có 0 cách
tổng cộng: 2


## 5. Cách 1 — Brute Force / baseline: bốn vòng lặp `O(n^4)`

**Ý tưởng.** Thử mọi bộ chỉ số `(i, j, k, l)`, đếm bộ nào có tổng `0`. Đúng hiển nhiên, nhưng `n^4`: với `n = 200` là `1.6 × 10^9` phép — Python cần hàng chục phút.

**Nút thắt.** Bốn vòng lặp lồng nhau trong khi điều kiện chỉ là **một đẳng thức**. Ta có thể "ghép đôi" để mỗi vế chỉ chứa hai mảng.

In [9]:
def four_sum_count_brute(nums1, nums2, nums3, nums4):
    n = len(nums1)
    count = 0
    for i in range(n):
        for j in range(n):
            for k in range(n):
                for l in range(n):
                    if nums1[i] + nums2[j] + nums3[k] + nums4[l] == 0:
                        count += 1
    return count


for arrays, expected in examples:
    r = four_sum_count_brute(*arrays)
    assert r == expected
    print(arrays, "->", r)

([1, 2], [-2, -1], [-1, 2], [0, 2]) -> 2
([0], [0], [0], [0]) -> 1


**Độ phức tạp.** Thời gian `O(n^4)`, bộ nhớ `O(1)`. Chỉ dùng làm oracle cho `n ≤ 10`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** `a + b + c + d = 0` ⇔ `a + b = -(c + d)`.

- Vế trái phụ thuộc chỉ `(i, j)`; vế phải chỉ `(k, l)` — hai nhóm **độc lập**.
- Số bộ `(i, j, k, l)` thoả = `Σ` theo mọi tổng `s` của **(số cặp `(i,j)` có `a+b = s`) × (số cặp `(k,l)` có `c+d = -s`)**.
- Chỉ cần bảng đếm `cnt[s]` của một nhóm (`n^2` cặp), rồi duyệt `n^2` cặp của nhóm kia và cộng `cnt[-(c+d)]`.

Giảm từ `n^4` xuống `n^2 + n^2 = 2n^2` phép (tra bảng băm `O(1)`). Với `n = 200`: `80000` phép — tức thời. Hãy xem sự chênh lệch bằng con số:

In [10]:
n = 200
print("n^4 =", n ** 4, "| 2*n^2 =", 2 * n ** 2, "| tỉ lệ ~", n ** 4 // (2 * n ** 2))

n^4 = 1600000000 | 2*n^2 = 80000 | tỉ lệ ~ 20000


## 7. Các cách tiếp theo

### Cách 2 — Ba vòng lặp + bảng đếm của `nums4` (`O(n^3)`)

**Ý tưởng trung gian.** Mới chỉ "gộp một mảng": đếm `cnt4 = Counter(nums4)`. Duyệt `(a, b, c)` (ba vòng) và cộng `cnt4[-(a + b + c)]` — số `d` thích hợp. Giảm một vòng lặp: `O(n^3)`; với `n = 200` là `8 × 10^6` phép tra bảng — chấp nhận được trong vài giây nhưng chưa tối ưu. Đây là bước đệm để thấy "mỗi lần gộp thêm một mảng vào bảng" giảm một bậc.

In [11]:
def four_sum_count_three_loops(nums1, nums2, nums3, nums4):
    cnt4 = Counter(nums4)
    count = 0
    for a in nums1:
        for b in nums2:
            for c in nums3:
                count += cnt4[-(a + b + c)]
    return count


for arrays, expected in examples:
    r = four_sum_count_three_loops(*arrays)
    assert r == expected
    print(arrays, "->", r)

([1, 2], [-2, -1], [-1, 2], [0, 2]) -> 2
([0], [0], [0], [0]) -> 1


**Độ phức tạp.** Thời gian `O(n^3)`, bộ nhớ `O(n)`.

### Cách 3 — **Gặp nhau ở giữa** bằng bảng băm các tổng cặp (`O(n^2)`) — cách chuẩn

**Giải thích từng đoạn.**

1. `cnt = Counter(a + b for a in nums1 for b in nums2)` — đếm mọi tổng cặp của hai mảng đầu (`n^2` tổng).
2. Với mỗi `c` trong `nums3`, mỗi `d` trong `nums4`: cộng `cnt[-(c + d)]` vào đáp án.
3. Trả đáp án.

Vì sao **cộng** (không phải nhân) ở bước 2? Mỗi cặp `(k, l)` cụ thể đóng góp đúng `cnt[-(c+d)]` bộ `(i, j)` tương ứng; các cặp `(k, l)` khác nhau là các bộ khác nhau.

In [12]:
def four_sum_count_meet(nums1, nums2, nums3, nums4):
    cnt = Counter(a + b for a in nums1 for b in nums2)
    return sum(cnt[-(c + d)] for c in nums3 for d in nums4)


for arrays, expected in examples:
    r = four_sum_count_meet(*arrays)
    assert r == expected
    print(arrays, "->", r)

([1, 2], [-2, -1], [-1, 2], [0, 2]) -> 2
([0], [0], [0], [0]) -> 1


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def fourSumCount(self, nums1: List[int], nums2: List[int], nums3: List[int], nums4: List[int]) -> int:
        cnt = Counter(a + b for a in nums1 for b in nums2)
        return sum(cnt[-(c + d)] for c in nums3 for d in nums4)
```

**Độ phức tạp.** Thời gian `O(n^2)` (xây bảng + `n^2` lần tra), bộ nhớ `O(n^2)` (tối đa `n^2` khoá khác nhau).

### Cách 4 — Hai bảng đếm rồi **nối theo khoá**: `Σ cntAB[s] × cntCD[-s]`

**Ý tưởng khác.** Thay vì duyệt `n^2` cặp của nhóm thứ hai, **cũng gom thành bảng đếm** `cntCD` rồi duyệt **các tổng phân biệt** của `cntAB`: với mỗi `s`, nhân `cntAB[s] × cntCD[-s]`. Đây là phép **nối (join)** hai bảng theo khoá `s` và `-s`; lợi khi có **nhiều tổng trùng nhau** (số tổng phân biệt ít hơn `n^2`).

**Giải thích từng đoạn.** `ab = Counter(...)`, `cd = Counter(...)`; `sum(v * cd[-s] for s, v in ab.items())`. Ở đây **nhân** vì mỗi tổng `s` có `ab[s]` cặp `(i,j)` và `cd[-s]` cặp `(k,l)`, ghép tuỳ ý → tích.

In [13]:
def four_sum_count_join(nums1, nums2, nums3, nums4):
    ab = Counter(a + b for a in nums1 for b in nums2)
    cd = Counter(c + d for c in nums3 for d in nums4)
    return sum(v * cd[-s] for s, v in ab.items())


for arrays, expected in examples:
    r = four_sum_count_join(*arrays)
    assert r == expected
    print(arrays, "->", r)

([1, 2], [-2, -1], [-1, 2], [0, 2]) -> 2
([0], [0], [0], [0]) -> 1


**Độ phức tạp.** Thời gian `O(n^2)`, bộ nhớ `O(n^2)` (hai bảng). Khi mọi phần tử bằng `0`, mỗi bảng chỉ có **một** khoá nên bước nối `O(1)`.

### Cách 5 — Sắp xếp hai danh sách tổng + **hai con trỏ** (`O(n^2 log n)`)

**Ý tưởng khác — không dùng bảng băm.** Lập danh sách các tổng cặp `AB` và `CD` (mỗi danh sách `n^2` phần tử, **giữ cả giá trị lặp**), sắp xếp tăng dần. Hai con trỏ: `i` đầu `AB`, `j` cuối `CD`:

- `AB[i] + CD[j] < 0` → `i += 1`; `> 0` → `j -= 1`.
- `== 0`: gặp giá trị `x = AB[i]`, `y = CD[j] = -x`. Đếm **số lần lặp** `ci` của `x` trong `AB` (đi `i` qua cả khối) và `cj` của `y` trong `CD`; cộng `ci × cj`.

**Điểm mấu chốt:** phải xử lý **khối giá trị lặp** bằng tích `ci × cj`; nếu chỉ `i += 1; j -= 1; ans += 1` sẽ **đếm thiếu** khi có tổng trùng.

In [14]:
def four_sum_count_two_pointers(nums1, nums2, nums3, nums4):
    ab = sorted(a + b for a in nums1 for b in nums2)
    cd = sorted(c + d for c in nums3 for d in nums4)
    i, j, ans = 0, len(cd) - 1, 0
    while i < len(ab) and j >= 0:
        s = ab[i] + cd[j]
        if s < 0:
            i += 1
        elif s > 0:
            j -= 1
        else:
            x, y = ab[i], cd[j]
            ci = cj = 0
            while i < len(ab) and ab[i] == x:
                i += 1
                ci += 1
            while j >= 0 and cd[j] == y:
                j -= 1
                cj += 1
            ans += ci * cj
    return ans


for arrays, expected in examples:
    r = four_sum_count_two_pointers(*arrays)
    assert r == expected
    print(arrays, "->", r)

([1, 2], [-2, -1], [-1, 2], [0, 2]) -> 2
([0], [0], [0], [0]) -> 1


**Độ phức tạp.** Sắp xếp `2` danh sách `n^2` phần tử: `O(n^2 log n)`; hai con trỏ `O(n^2)`; bộ nhớ `O(n^2)`. Chậm hơn bảng băm một hệ số `log` nhưng **không phụ thuộc băm** (không có trường hợp xấu về băm) và cho thấy một họ kỹ thuật khác (sắp xếp + hai con trỏ như 3Sum/4Sum).

## 8. TRACE từng bước

Ta trace **Cách 3** (bảng băm) và **Cách 5** (hai con trỏ) trên Ví dụ 1.

In [15]:
def trace_meet(nums1, nums2, nums3, nums4):
    cnt = Counter()
    print("Giai đoạn 1: xây bảng tổng a+b")
    for a in nums1:
        for b in nums2:
            cnt[a + b] += 1
            print(f"  a={a:>2}, b={b:>2}: a+b={a + b:>2} -> cnt = {dict(cnt)}")
    print("Giai đoạn 2: duyệt c, d và tra -(c+d)")
    total = 0
    for c in nums3:
        for d in nums4:
            need = -(c + d)
            hit = cnt[need]
            total += hit
            print(f"  c={c:>2}, d={d:>2}: c+d={c + d:>2}, cần {need:>2} -> cnt[{need}] = {hit}, tổng = {total}")
    return total


assert trace_meet([1, 2], [-2, -1], [-1, 2], [0, 2]) == 2

Giai đoạn 1: xây bảng tổng a+b
  a= 1, b=-2: a+b=-1 -> cnt = {-1: 1}
  a= 1, b=-1: a+b= 0 -> cnt = {-1: 1, 0: 1}
  a= 2, b=-2: a+b= 0 -> cnt = {-1: 1, 0: 2}
  a= 2, b=-1: a+b= 1 -> cnt = {-1: 1, 0: 2, 1: 1}
Giai đoạn 2: duyệt c, d và tra -(c+d)
  c=-1, d= 0: c+d=-1, cần  1 -> cnt[1] = 1, tổng = 1
  c=-1, d= 2: c+d= 1, cần -1 -> cnt[-1] = 1, tổng = 2
  c= 2, d= 0: c+d= 2, cần -2 -> cnt[-2] = 0, tổng = 2
  c= 2, d= 2: c+d= 4, cần -4 -> cnt[-4] = 0, tổng = 2


In [16]:
def trace_two_pointers(nums1, nums2, nums3, nums4):
    ab = sorted(a + b for a in nums1 for b in nums2)
    cd = sorted(c + d for c in nums3 for d in nums4)
    print("AB đã sắp:", ab)
    print("CD đã sắp:", cd)
    i, j, ans = 0, len(cd) - 1, 0
    while i < len(ab) and j >= 0:
        s = ab[i] + cd[j]
        print(f"i={i}, j={j}: AB[i]={ab[i]}, CD[j]={cd[j]}, tổng={s}", end=" -> ")
        if s < 0:
            print("nhỏ quá, i += 1")
            i += 1
        elif s > 0:
            print("lớn quá, j -= 1")
            j -= 1
        else:
            x, y = ab[i], cd[j]
            ci = cj = 0
            while i < len(ab) and ab[i] == x:
                i += 1; ci += 1
            while j >= 0 and cd[j] == y:
                j -= 1; cj += 1
            ans += ci * cj
            print(f"bằng 0: {x} xuất hiện {ci} lần, {y} xuất hiện {cj} lần -> += {ci}*{cj}, ans = {ans}")
    return ans


assert trace_two_pointers([1, 2], [-2, -1], [-1, 2], [0, 2]) == 2

AB đã sắp: [-1, 0, 0, 1]
CD đã sắp: [-1, 1, 2, 4]
i=0, j=3: AB[i]=-1, CD[j]=4, tổng=3 -> lớn quá, j -= 1
i=0, j=2: AB[i]=-1, CD[j]=2, tổng=1 -> lớn quá, j -= 1
i=0, j=1: AB[i]=-1, CD[j]=1, tổng=0 -> bằng 0: -1 xuất hiện 1 lần, 1 xuất hiện 1 lần -> += 1*1, ans = 1
i=1, j=0: AB[i]=0, CD[j]=-1, tổng=-1 -> nhỏ quá, i += 1
i=2, j=0: AB[i]=0, CD[j]=-1, tổng=-1 -> nhỏ quá, i += 1
i=3, j=0: AB[i]=1, CD[j]=-1, tổng=0 -> bằng 0: 1 xuất hiện 1 lần, -1 xuất hiện 1 lần -> += 1*1, ans = 2


Bảng trace Cách 3:

| Giai đoạn | Bước | Trạng thái |
|---|---|---|
| 1 | `(1,-2)` | `cnt = {-1: 1}` |
| 1 | `(1,-1)` | `cnt = {-1: 1, 0: 1}` |
| 1 | `(2,-2)` | `cnt = {-1: 1, 0: 2}` |
| 1 | `(2,-1)` | `cnt = {-1: 1, 0: 2, 1: 1}` |
| 2 | `(c,d) = (-1,0)` | `-(c+d) = 1` → `cnt[1] = 1`, tổng `1` |
| 2 | `(-1,2)` | `-1` → `cnt[-1] = 1`, tổng `2` |
| 2 | `(2,0)` | `-2` → `0`, tổng `2` |
| 2 | `(2,2)` | `-4` → `0`, tổng `2` |

Kết quả `2` ✓. Ở Cách 5, `AB = [-1, 0, 0, 1]` và `CD = [-1, 1, 2, 4]`: hai con trỏ gặp cặp `(-1, 1)` rồi `(1, -1)`.

## 9. Code nộp LeetCode

Bản nộp dùng **Cách 3** (bảng băm các tổng cặp): `O(n^2)` thời gian và bộ nhớ.

In [17]:
class Solution:
    def fourSumCount(self, nums1: List[int], nums2: List[int], nums3: List[int], nums4: List[int]) -> int:
        cnt = Counter(a + b for a in nums1 for b in nums2)
        return sum(cnt[-(c + d)] for c in nums3 for d in nums4)


sol = Solution()
for arrays, expected in examples:
    r = sol.fourSumCount(*arrays)
    assert r == expected
    print(arrays, "->", r)

([1, 2], [-2, -1], [-1, 2], [0, 2]) -> 2
([0], [0], [0], [0]) -> 1


## 10. Vì sao thuật toán đúng

Đáp án `= |{(i,j,k,l) : a_i + b_j + c_k + d_l = 0}|`, trong đó `a, b, c, d` là bốn mảng.

**Phân hoạch theo cặp `(k, l)`.** Mỗi bộ `(i,j,k,l)` thuộc đúng **một** nhóm ứng với cặp `(k, l)` của nó. Với cặp `(k, l)` cố định, các `(i, j)` hợp lệ là những cặp có `a_i + b_j = -(c_k + d_l)`; số này bằng `cnt[-(c_k + d_l)]` theo định nghĩa của `cnt` (bảng đếm các tổng `a_i + b_j` trên **mọi** cặp `(i, j)`, cả khi trùng giá trị). Cộng theo mọi `(k, l)` được tổng. ∎

**Vì sao `Counter` đếm đúng "chỉ số":** `Counter(generator)` tăng đếm **một lần cho mỗi phần tử** do generator sinh ra, nên hai cặp chỉ số khác nhau có cùng tổng được tính là hai lần.

Kiểm chứng bằng vét cạn và bằng cách nhìn theo hướng ngược lại (đếm theo cặp `(i, j)` thay vì `(k, l)`): hai hướng phải cho cùng kết quả.

In [18]:
def four_sum_count_other_side(nums1, nums2, nums3, nums4):
    """Đối xứng: bảng của (c+d), duyệt (a, b) -- phải cho cùng đáp án."""
    cnt = Counter(c + d for c in nums3 for d in nums4)
    return sum(cnt[-(a + b)] for a in nums1 for b in nums2)


rng = random.Random(454)
for _ in range(300):
    n = rng.randint(1, 6)
    arrays = tuple([rng.randint(-3, 3) for _ in range(n)] for _ in range(4))
    brute = sum(1 for t in product(*arrays) if sum(t) == 0)       # oracle độc lập (product)
    assert four_sum_count_meet(*arrays) == brute == four_sum_count_other_side(*arrays)
print("300 input nhỏ: bảng băm (hai hướng) = vét cạn bằng product.")

300 input nhỏ: bảng băm (hai hướng) = vét cạn bằng product.


## 11. Phân tích Time / Space Complexity

Với `n` là độ dài mỗi mảng:

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Bốn vòng lặp | `O(n^4)` | `O(1)` | chỉ làm oracle |
| 2. Ba vòng + `Counter(nums4)` | `O(n^3)` | `O(n)` | ~8·10^6 phép với `n = 200` |
| 3. Bảng tổng cặp (gặp nhau ở giữa) | `O(n^2)` | `O(n^2)` | **cách nộp bài** |
| 4. Hai bảng + nối theo khoá | `O(n^2)` | `O(n^2)` | tốt khi nhiều tổng trùng |
| 5. Sắp xếp + hai con trỏ | `O(n^2 log n)` | `O(n^2)` | không dựa vào băm |

Với `n = 200`: Cách 3 dùng `40 000` khoá và `40 000` lần tra — vài chục mili-giây.

## 12. Bộ test

### 12.1. Ví dụ của đề, đối chiếu các cách

In [19]:
METHODS = {
    "three_loops": four_sum_count_three_loops,
    "meet": four_sum_count_meet,
    "join": four_sum_count_join,
    "two_pointers": four_sum_count_two_pointers,
    "Solution": Solution().fourSumCount,
}


def oracle(nums1, nums2, nums3, nums4):
    return sum(1 for t in product(nums1, nums2, nums3, nums4) if sum(t) == 0)


for arrays, expected in examples:
    assert oracle(*arrays) == expected
    assert four_sum_count_brute(*arrays) == expected
    for name, fn in METHODS.items():
        assert fn(*arrays) == expected, (name, arrays)
print("2 ví dụ của đề: tất cả các cách đều đúng.")

2 ví dụ của đề: tất cả các cách đều đúng.


### 12.2. Edge case

`n = 1` (có/không có nghiệm), toàn `0` (đáp án `n^4`), toàn dương (đáp án `0`), toàn âm, các phần tử bằng `±2^28` ở biên, giá trị trùng nhiều, và cân bằng chính xác.

In [20]:
M = 2 ** 28
edge_cases = [
    (([1], [1], [1], [-3]), 1),
    (([1], [1], [1], [1]), 0),
    (([0] * 5, [0] * 5, [0] * 5, [0] * 5), 5 ** 4),            # mọi bộ đều có tổng 0
    (([1, 2, 3], [1, 2, 3], [1, 2, 3], [1, 2, 3]), 0),          # toàn dương
    (([-1, -2], [-3, -4], [-5, -6], [-7, -8]), 0),              # toàn âm
    (([M], [M], [-M], [-M]), 1),                                 # biên 2^28
    (([M, -M], [M, -M], [M, -M], [M, -M]), 6),                   # C(4,2) = 6 cách chọn hai dấu +
    (([1, 1, 1], [-1, -1], [2, 2], [-2, -2]), 3 * 2 * 2 * 2),   # mọi bộ đều cân bằng: 1-1+2-2 = 0
    (([0, 0], [1, -1], [-1, 1], [0]), 4),                       # a = 0 (2 cách), (b, c) = (1,-1) hoặc (-1,1)
]
for arrays, expected in edge_cases:
    assert oracle(*arrays) == expected, arrays
    for name, fn in METHODS.items():
        assert fn(*arrays) == expected, (name, arrays)
print(f"{len(edge_cases)} edge case: tất cả các cách đều đúng.")

9 edge case: tất cả các cách đều đúng.


### 12.3. Phản ví dụ cho các cách làm sai

Kiểm tra rằng các cách sai phổ biến **bị bắt** bởi test:

- (a) Dùng `set` thay `Counter` (mất số lần).
- (b) Loại trùng giá trị (kiểu 4Sum gốc).
- (c) Quên dấu trừ khi tra `cnt[c + d]`.

In [21]:
def wrong_set(nums1, nums2, nums3, nums4):
    ab = {a + b for a in nums1 for b in nums2}          # SAI: set mất số lần
    return sum(1 for c in nums3 for d in nums4 if -(c + d) in ab)


def wrong_dedupe(nums1, nums2, nums3, nums4):
    s1, s2, s3, s4 = map(set, (nums1, nums2, nums3, nums4))   # SAI: bỏ trùng giá trị
    ab = Counter(a + b for a in s1 for b in s2)
    return sum(ab[-(c + d)] for c in s3 for d in s4)


def wrong_sign(nums1, nums2, nums3, nums4):
    ab = Counter(a + b for a in nums1 for b in nums2)
    return sum(ab[c + d] for c in nums3 for d in nums4)       # SAI: thiếu dấu trừ


cases = [
    (([0, 0], [0], [0], [0]), 2),
    (([1, 2], [-2, -1], [-1, 2], [0, 2]), 2),
    (([1], [1], [1], [-3]), 1),
]
for fn in (wrong_set, wrong_dedupe, wrong_sign):
    outs = [(fn(*arr), exp) for arr, exp in cases]
    bad = [o for o in outs if o[0] != o[1]]
    print(f"{fn.__name__:13s}: (kết quả, đúng) = {outs} -> {'bị bắt' if bad else 'KHÔNG bị bắt'}")
    assert bad

wrong_set    : (kết quả, đúng) = [(1, 2), (2, 2), (1, 1)] -> bị bắt
wrong_dedupe : (kết quả, đúng) = [(1, 2), (2, 2), (1, 1)] -> bị bắt
wrong_sign   : (kết quả, đúng) = [(2, 2), (2, 2), (0, 1)] -> bị bắt


### 12.4. Đối chiếu ngẫu nhiên (có seed) với oracle vét cạn

Input nhỏ `n ≤ 7`, giá trị trong `[-4, 4]` (để có nhiều tổng bằng nhau và nhiều nghiệm); cả hai "cách" vét cạn (4 vòng và `product`) và mọi cách nhanh phải trùng.

In [22]:
rng = random.Random(4540)
for trial in range(500):
    n = rng.randint(1, 7)
    span = rng.choice([1, 2, 4])
    arrays = tuple([rng.randint(-span, span) for _ in range(n)] for _ in range(4))
    expected = four_sum_count_brute(*arrays)
    assert expected == oracle(*arrays)
    for name, fn in METHODS.items():
        assert fn(*arrays) == expected, (name, arrays, expected)
print("500 input ngẫu nhiên nhỏ: mọi cách = vét cạn.")

500 input ngẫu nhiên nhỏ: mọi cách = vét cạn.


### 12.5. Đối chiếu có "nghiệm cài sẵn" và giá trị lớn

Với `n` đến `40` và giá trị đến `±2^28` ngẫu nhiên, ít khi có nghiệm; nên ta **cài sẵn nghiệm** bằng cách đặt `nums4[l] = -(nums1[i] + nums2[j] + nums3[k])` cho vài bộ (bảo đảm `-2^28 <= ... <= 2^28`). Các cách `O(n^3)`, `O(n^2)` phải khớp nhau.

In [23]:
rng = random.Random(45400)
M = 2 ** 28
for trial in range(40):
    n = rng.randint(2, 40)
    big = rng.choice([3, 50, M // 4])
    a = [rng.randint(-big, big) for _ in range(n)]
    b = [rng.randint(-big, big) for _ in range(n)]
    c = [rng.randint(-big, big) for _ in range(n)]
    d = [rng.randint(-big, big) for _ in range(n)]
    for _ in range(rng.randint(0, n)):          # cài nghiệm
        l = rng.randrange(n)
        d[l] = -(a[rng.randrange(n)] + b[rng.randrange(n)] + c[rng.randrange(n)])
        d[l] = max(-M, min(M, d[l]))
    ref = four_sum_count_three_loops(a, b, c, d)
    for name, fn in METHODS.items():
        assert fn(a, b, c, d) == ref, (trial, name)
print("40 input vừa với nghiệm cài sẵn: mọi cách khớp nhau.")

40 input vừa với nghiệm cài sẵn: mọi cách khớp nhau.


### 12.6. Hiệu năng ở giới hạn `n = 200`

In [24]:
rng = random.Random(7)
n = 200
rand_case = tuple([rng.randint(-M, M) for _ in range(n)] for _ in range(4))
small_case = tuple([rng.randint(-10, 10) for _ in range(n)] for _ in range(4))       # rất nhiều tổng trùng
zero_case = tuple([0] * n for _ in range(4))                                          # đáp án n^4

for label, arrays in [("ngẫu nhiên lớn", rand_case), ("giá trị nhỏ (nhiều trùng)", small_case), ("toàn 0", zero_case)]:
    print(f"--- {label}")
    results = {}
    for name in ["meet", "join", "two_pointers", "Solution"]:
        t0 = time.perf_counter()
        results[name] = METHODS[name](*arrays)
        print(f"  {name:13s} {time.perf_counter() - t0:.3f}s -> {results[name]}")
    assert len(set(results.values())) == 1
assert METHODS["meet"](*zero_case) == 200 ** 4
t0 = time.perf_counter()
r = four_sum_count_three_loops(*small_case)
print(f"  three_loops   {time.perf_counter() - t0:.3f}s -> {r} (O(n^3))")
assert r == METHODS["meet"](*small_case)

--- ngẫu nhiên lớn
  meet          0.011s -> 1
  join          0.015s -> 1
  two_pointers  0.023s -> 1
  Solution      0.009s -> 1
--- giá trị nhỏ (nhiều trùng)
  meet          0.005s -> 49492247
  join          0.005s -> 49492247
  two_pointers  0.012s -> 49492247
  Solution      0.005s -> 49492247
--- toàn 0
  meet          0.004s -> 1600000000
  join          0.004s -> 1600000000
  two_pointers  0.007s -> 1600000000
  Solution      0.004s -> 1600000000


  three_loops   0.905s -> 49492247 (O(n^3))


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `set` thay `Counter` (mất số lần)

In [25]:
arrays = ([0, 0], [0], [0], [0])
print("sai :", wrong_set(*arrays), "| đúng:", Solution().fourSumCount(*arrays))
assert wrong_set(*arrays) == 1 and Solution().fourSumCount(*arrays) == 2

sai : 1 | đúng: 2


**Vì sao sai:** `{0, 0}` thu về `{0}` — hai cặp chỉ số có cùng tổng chỉ tính một. Đề đếm **bộ chỉ số**. **Đúng:** `Counter` (hoặc `dict` đếm).

### Lỗi 2: Loại trùng giá trị như bài 4Sum gốc

In [26]:
arrays = ([1, 1], [-1], [0], [0])
print("sai :", wrong_dedupe(*arrays), "| đúng:", Solution().fourSumCount(*arrays))
assert wrong_dedupe(*arrays) == 1 and Solution().fourSumCount(*arrays) == 2

sai : 1 | đúng: 2


**Vì sao sai:** `nums1 = [1, 1]` có hai chỉ số `i = 0, 1` cho hai bộ khác nhau. Bài này **không** cần các bộ giá trị phân biệt.

### Lỗi 3: Quên dấu trừ khi tra bảng

In [27]:
arrays = ([1], [1], [1], [-3])
print("sai :", wrong_sign(*arrays), "| đúng:", Solution().fourSumCount(*arrays))
assert wrong_sign(*arrays) == 0 and Solution().fourSumCount(*arrays) == 1

sai : 0 | đúng: 1


**Vì sao sai:** ta cần `a + b = -(c + d)`; tra `cnt[c + d]` tìm `a + b = c + d` — sai điều kiện.

### Lỗi 4: Dùng `defaultdict` rồi tra khoá không có (bảng phình ra)

In [28]:
def count_with_defaultdict(nums1, nums2, nums3, nums4):
    cnt = defaultdict(int)
    for a in nums1:
        for b in nums2:
            cnt[a + b] += 1
    total = sum(cnt[-(c + d)] for c in nums3 for d in nums4)      # mỗi lần tra khoá lạ lại THÊM khoá
    return total, len(cnt)


rng = random.Random(1)
arrays = tuple([rng.randint(-M, M) for _ in range(60)] for _ in range(4))
total, size = count_with_defaultdict(*arrays)
print("kết quả:", total, "| kích thước bảng sau khi tra:", size, "(ban đầu tối đa", 60 * 60, ")")
assert total == Solution().fourSumCount(*arrays)
assert size > len(Counter(a + b for a in arrays[0] for b in arrays[1]))

kết quả: 0 | kích thước bảng sau khi tra: 7200 (ban đầu tối đa 3600 )


**Vì sao nên tránh:** kết quả vẫn đúng nhưng bảng **phình thêm** tới `n^2` khoá rác (bộ nhớ gấp đôi). **Đúng:** `Counter` (tra không thêm), hoặc `cnt.get(key, 0)`.

### Lỗi 5: Hai con trỏ nhưng không xử lý khối giá trị lặp

In [29]:
def wrong_two_pointers(nums1, nums2, nums3, nums4):
    ab = sorted(a + b for a in nums1 for b in nums2)
    cd = sorted(c + d for c in nums3 for d in nums4)
    i, j, ans = 0, len(cd) - 1, 0
    while i < len(ab) and j >= 0:
        s = ab[i] + cd[j]
        if s < 0:
            i += 1
        elif s > 0:
            j -= 1
        else:
            ans += 1                     # SAI: mỗi lần gặp chỉ cộng 1, bỏ qua các bản sao
            i += 1
            j -= 1
    return ans


arrays = ([0, 0], [0, 0], [0, 0], [0, 0])
print("sai :", wrong_two_pointers(*arrays), "| đúng:", four_sum_count_two_pointers(*arrays), "(16 = 2^4)")
assert wrong_two_pointers(*arrays) == 4 and four_sum_count_two_pointers(*arrays) == 16

sai : 4 | đúng: 16 (16 = 2^4)


**Vì sao sai:** có `4` tổng `0` ở mỗi danh sách (`ab = [0,0,0,0]`, `cd = [0,0,0,0]`); số cặp đúng là `4 × 4 = 16`, nhưng cộng `1` và nhích cả hai con trỏ chỉ đếm `4`. **Đúng:** đếm cả khối lặp rồi cộng `ci × cj` (Cách 5).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Tổng của 4 số = 0, mỗi số từ một mảng khác nhau"** → **gặp nhau ở giữa**: tách thành `(2 số) + (2 số)`, bảng băm cho nửa đầu, duyệt nửa sau.
- Nguyên tắc chung: nếu vế trái và vế phải **độc lập** thì tính một vế, lưu lại, rồi tra cho vế kia (đổi tích `n^4` thành tổng `n^2 + n^2`).
- Đề cho **bốn mảng riêng biệt** → không cần sắp xếp/ loại trùng (khác bài 4Sum, 18). Đếm theo chỉ số.
- Cần "đếm số lần" → `Counter`; chỉ cần "có hay không" → `set`.
- Tra khoá không tồn tại: `Counter`/`get(k, 0)`; tránh `defaultdict` ở bước tra.
- Kết quả có thể rất lớn (`n^4`); Python không tràn số, ở C++/Java dùng `long` khi cộng (đáp án đến `1.6 × 10^9` vẫn nằm trong `int32` nhưng nên cẩn thận ở biến thể lớn hơn).

## 15. Biến thể của bài

1. **2Sum / 3Sum / 4Sum** (1, 15, 18): một mảng, trả về các bộ **giá trị phân biệt** → sắp xếp + hai con trỏ + loại trùng.
2. **Đích khác `0`**: tổng bằng `target` → tra `target - (c + d)`.
3. **Nhiều mảng hơn** (6 mảng, 3 + 3): gặp nhau ở giữa với bảng của 3 mảng: `O(n^3)`.
4. **Tổng nhỏ hơn / lớn hơn `target`** (bài 3Sum Smaller): không còn dùng bảng băm đơn giản; cần sắp xếp + hai con trỏ/ nhị phân.
5. **Đếm bộ chỉ số trong cùng một mảng** (`nums[i] + nums[j] = nums[k] + nums[l]`): cũng "gặp nhau ở giữa" với bảng các tổng cặp.

Ví dụ biến thể 2 (đích `target` bất kỳ):

In [30]:
def four_sum_count_target(nums1, nums2, nums3, nums4, target):
    cnt = Counter(a + b for a in nums1 for b in nums2)
    return sum(cnt[target - (c + d)] for c in nums3 for d in nums4)


rng = random.Random(15)
for _ in range(200):
    n = rng.randint(1, 5)
    arrays = tuple([rng.randint(-3, 3) for _ in range(n)] for _ in range(4))
    t = rng.randint(-6, 6)
    assert four_sum_count_target(*arrays, t) == sum(1 for x in product(*arrays) if sum(x) == t)
assert four_sum_count_target([1, 2], [-2, -1], [-1, 2], [0, 2], 0) == 2
print("Biến thể target: OK")

Biến thể target: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Bốn vòng | thử mọi bộ chỉ số | `O(n^4)` | `O(1)` | oracle, `n ≤ 10` |
| 2. Ba vòng + `Counter(nums4)` | gộp một mảng | `O(n^3)` | `O(n)` | bước đệm; `n` nhỏ/vừa |
| 3. Bảng tổng cặp | `a+b = -(c+d)`, tra bảng | `O(n^2)` | `O(n^2)` | **nộp bài** |
| 4. Hai bảng + nối | `Σ cntAB[s]·cntCD[-s]` | `O(n^2)` | `O(n^2)` | nhiều tổng trùng nhau |
| 5. Sắp xếp + hai con trỏ | hai danh sách tổng đã sắp | `O(n^2 log n)` | `O(n^2)` | không muốn dùng băm |

Quy tắc chọn: nộp bài → Cách 3; hỏi "không dùng băm?" → Cách 5.

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `Counter([1, 2, 2, 3])[2]`, `Counter(a + b for a in [0, 1] for b in [0, 1])`, `four_sum_count_meet([0], [0], [0], [0])`, `four_sum_count_meet([1], [1], [1], [1])`, `four_sum_count_meet([1, -1], [0], [0], [0])`.

**Bài 2 (tính tay).** Với `nums1 = [-1, -1]`, `nums2 = [-1, 1]`, `nums3 = [-1, 1]`, `nums4 = [1, 1]`: (a) lập bảng đếm tổng `a + b`; (b) tra từng `(c, d)` và cộng; (c) đối chiếu bằng cách liệt kê vài bộ chỉ số. Đáp án?

**Bài 3 (điền TODO).** Hoàn thành `four_sum_count_todo(nums1, nums2, nums3, nums4)` theo Cách 3.

In [31]:
def four_sum_count_todo(nums1, nums2, nums3, nums4):
    # TODO 1: cnt = Counter(a + b for a in nums1 for b in nums2)
    # TODO 2: total = 0; duyệt c trong nums3, d trong nums4: total += cnt[-(c + d)]
    # TODO 3: return total
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `four_sum_count_balanced(nums1, nums2, nums3, nums4)` dùng **hai bảng đếm nhưng duyệt khoá của bảng NHỎ HƠN** (so `len(ab)` và `len(cd)`), để tối ưu khi một bên có ít tổng phân biệt. Kết quả phải bằng Cách 3.

**Bài 5 (biến thể / mini challenge).** (a) Viết `six_sum_count(a, b, c, d, e, f)` đếm số bộ chỉ số có tổng `0` của **sáu** mảng, bằng cách gặp nhau ở giữa `3 + 3` (độ phức tạp `O(n^3)`). (b) Viết `count_pairs_equal_sums(nums)` đếm số bộ chỉ số `(i, j, k, l)` trong **một** mảng với `nums[i] + nums[j] == nums[k] + nums[l]` (cho phép các chỉ số lặp). Kiểm tra bằng vét cạn.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `2`, `Counter({0: 1, 1: 2, 2: 1})`, `1`, `0`, `0`. (với `[1, -1], [0], [0], [0]`: tổng là `1` hoặc `-1`, không bằng `0`.) Bài 3: bám sát mã Cách 3 ở mục 7.
- **Mức 2 (vừa):** Bài 2: bảng `a+b = {-2: 2, 0: 2}`. Bốn cặp `(c, d)`: hai cặp có `c+d = 0` (cần `0`: `2` cách mỗi cặp) và hai cặp có `c+d = 2` (cần `-2`: `2` cách mỗi cặp) → `2·2 + 2·2 = 8`. Bài 4: `if len(ab) > len(cd): ab, cd = cd, ab` rồi `sum(v * cd[-s] for s, v in ab.items())` (đối xứng vì `-(-s)`).
- **Mức 3 (sâu):** Bài 5(a): `abc = Counter(x + y + z for ...)`, `def_ = Counter(...)`; nối theo khoá. 5(b): `Counter(a + b for a in nums for b in nums)`, rồi `sum(v * v for v in cnt.values())` (mỗi tổng `s` có `v` cặp `(i, j)` và `v` cặp `(k, l)`).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert Counter([1, 2, 2, 3])[2] == 2
assert Counter(a + b for a in [0, 1] for b in [0, 1]) == Counter({0: 1, 1: 2, 2: 1})
assert four_sum_count_meet([0], [0], [0], [0]) == 1
assert four_sum_count_meet([1], [1], [1], [1]) == 0
assert four_sum_count_meet([1, -1], [0], [0], [0]) == 0

# Bài 2
arrays2 = ([-1, -1], [-1, 1], [-1, 1], [1, 1])
ab2 = Counter(a + b for a in arrays2[0] for b in arrays2[1])
print("Bài 2 (a): bảng a+b =", dict(ab2))
total2 = sum(ab2[-(c + d)] for c in arrays2[2] for d in arrays2[3])
print("Bài 2: đáp án =", total2)
assert total2 == oracle(*arrays2) == four_sum_count_brute(*arrays2)
assert dict(ab2) == {-2: 2, 0: 2} and total2 == 8


# Bài 3
def four_sum_count_todo_solution(nums1, nums2, nums3, nums4):
    cnt = Counter(a + b for a in nums1 for b in nums2)
    total = 0
    for c in nums3:
        for d in nums4:
            total += cnt[-(c + d)]
    return total


# Bài 4
def four_sum_count_balanced(nums1, nums2, nums3, nums4):
    ab = Counter(a + b for a in nums1 for b in nums2)
    cd = Counter(c + d for c in nums3 for d in nums4)
    if len(ab) > len(cd):
        ab, cd = cd, ab                       # duyệt bảng nhỏ hơn, tra bảng lớn hơn
    return sum(v * cd[-s] for s, v in ab.items())


# Bài 5a
def six_sum_count(a, b, c, d, e, f):
    left = Counter(x + y + z for x in a for y in b for z in c)
    right = Counter(x + y + z for x in d for y in e for z in f)
    return sum(v * right[-s] for s, v in left.items())


# Bài 5b
def count_pairs_equal_sums(nums):
    cnt = Counter(a + b for a in nums for b in nums)
    return sum(v * v for v in cnt.values())


rng = random.Random(1919)
for _ in range(300):
    n = rng.randint(1, 6)
    arrays = tuple([rng.randint(-3, 3) for _ in range(n)] for _ in range(4))
    exp = oracle(*arrays)
    assert four_sum_count_todo_solution(*arrays) == exp
    assert four_sum_count_balanced(*arrays) == exp
    m = rng.randint(1, 4)
    six = tuple([rng.randint(-2, 2) for _ in range(m)] for _ in range(6))
    assert six_sum_count(*six) == sum(1 for t in product(*six) if sum(t) == 0)
    nums = [rng.randint(-3, 3) for _ in range(rng.randint(1, 6))]
    brute = sum(1 for i, j, k, l in product(range(len(nums)), repeat=4) if nums[i] + nums[j] == nums[k] + nums[l])
    assert count_pairs_equal_sums(nums) == brute
print("Tất cả đáp án bài tập đều đúng.")

Bài 2 (a): bảng a+b = {-2: 2, 0: 2}
Bài 2: đáp án = 8
Tất cả đáp án bài tập đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm bộ có tổng bằng đích bằng "gặp nhau ở giữa")

```python
class Solution:
    def fourSumCount(self, nums1: List[int], nums2: List[int], nums3: List[int], nums4: List[int]) -> int:
        cnt = Counter(a + b for a in nums1 for b in nums2)       # nửa 1: đếm mọi tổng cặp
        return sum(cnt[-(c + d)] for c in nums3 for d in nums4)  # nửa 2: tra phần bù
```

**Checklist**

- [ ] Đếm theo **bộ chỉ số** (không loại trùng giá trị, không sắp xếp để loại trùng)?
- [ ] Dùng `Counter` (đếm số lần), không phải `set`?
- [ ] Tra `-(c + d)` (có dấu trừ) — hoặc `target - (c + d)` nếu đích khác `0`?
- [ ] Tra bảng không làm phình bảng (`Counter`/`get`, tránh `defaultdict`)?
- [ ] Độ phức tạp `O(n^2)`: `n = 200` → `40 000` khoá, rất nhanh?
- [ ] Đã thử: `n = 1`, toàn `0` (đáp án `n^4`), toàn dương/ toàn âm, biên `±2^28`, giá trị trùng nhiều?
- [ ] Skeleton đúng `class Solution` / `def fourSumCount(self, nums1, nums2, nums3, nums4) -> int`?